# The goal is to collect all the data form the different years and compare the best algos from each year to determine the best of the best : BBOB test Suite

# 1) Collecting the data from the different years and making one single file with all the best algos from all the different years.

We first build a "table" where we will agregate all the data from all of the different years.

In [19]:
import pandas as pd
from pathlib import Path

# === 1. Load all yearly CSVs from the "results" folder ===
folder = Path("results")
all_files = sorted(folder.glob("best_algos_*.csv"))

dfs = []
for f in all_files:
    # Extract year (e.g. "best_algos_2009.csv" → 2009)
    year = int(f.stem.split("_")[-1])
    df = pd.read_csv(f)
    df["year"] = year
    dfs.append(df)

# Merge everything into a single DataFrame
df_all = pd.concat(dfs, ignore_index=True)

# === 2. Find, for each (dim, func, target), the entry with smallest ERT ===
idx = df_all.groupby(["dimension", "function_id", "target"])["best_ERT"].idxmin()
df_best_overall = df_all.loc[idx, ["dimension", "function_id", "year", "best_algorithm", "target", "best_ERT"]]

# Sort nicely
df_best_overall = df_best_overall.sort_values(["dimension", "function_id", "target"]).reset_index(drop=True)

# === 3. Display result ===
print("✅ Global best algorithm across all years (smallest ERT for each dim/function/target):\n")
print(df_best_overall.head(20))

# Optionally save to file
df_best_overall.to_csv("results/global_best_algos.csv", index=False)


✅ Global best algorithm across all years (smallest ERT for each dim/function/target):

    dimension  function_id  year              best_algorithm        target  \
0           2            1  2010             HCMA_loshchilov  1.000000e-08   
1           2            1  2010             HCMA_loshchilov  1.000000e-05   
2           2            1  2010             HCMA_loshchilov  1.000000e-03   
3           2            1  2010             HCMA_loshchilov  1.000000e-02   
4           2            1  2009                  NEWUOA_ros  1.000000e-01   
5           2            2  2009             LSfminbnd_posik  1.000000e-08   
6           2            2  2009             LSfminbnd_posik  1.000000e-05   
7           2            2  2009             LSfminbnd_posik  1.000000e-03   
8           2            2  2009             LSfminbnd_posik  1.000000e-02   
9           2            2  2009             LSfminbnd_posik  1.000000e-01   
10          2            3  2009                LSstep_

# 2) General table summing up nice stats about the best algos

- Where the best algos are located (year)
- Which algorithms are the dominant winners (counter)

This wil be used to check the results especially for the count.

In [20]:
# Count wins per year
print(df_best_overall["year"].value_counts())

# Count wins per algorithm
print(df_best_overall["best_algorithm"].value_counts().head(10))


2010    196
2009    111
2020    111
2012     67
2019     61
2023     47
2022     44
2021     37
2018     28
2017     17
2016      1
Name: year, dtype: int64
SLSQP+lq-CMA-ES_Hansen           37
BIPOP-aCMA-STEP_loshchilov       35
SLSQP-scipy-2019_Varelas         34
HMLSL_pal                        33
HE-ES_Glasmachers                31
LSfminbnd_posik                  30
BIPOP-saACM-k_loshchilov         29
SHADE-LM-POP4-to-10_Okulewicz    26
OQNLP_pal                        24
fminunc_pal                      23
Name: best_algorithm, dtype: int64


We can see that the year where there are the most "best algos" is the year 2010. 
in addition the algo that appears the most is SLSQP+lq-CMA-ES_Hansen.

In [21]:
# === 4. Aggregate: count how many times each algorithm was best per dimension ===

# Count how many times each algorithm appears as best within each dimension
algo_counts = (
    df_best_overall
    .groupby(["dimension", "best_algorithm"])
    .size()  # count occurrences
    .reset_index(name="count")
)

# Sort within each dimension by count descending
algo_counts = (
    algo_counts
    .sort_values(["dimension", "count"], ascending=[True, False])
)

# For convenience: for each dimension, keep rank of algorithms by count
algo_counts["rank"] = algo_counts.groupby("dimension")["count"].rank(method="first", ascending=False)


pivot_table = algo_counts.pivot(index="best_algorithm", columns="dimension", values="count").fillna(0).astype(int)
pivot_table["Total"] = pivot_table.sum(axis=1)
pivot_table = pivot_table.sort_values("Total", ascending=False)

print("\n Overall frequency of being best by algorithm and dimension (top 10):\n")
print(pivot_table.head(10))



 Overall frequency of being best by algorithm and dimension (top 10):

dimension                       2  3   5  10  20  40  Total
best_algorithm                                             
SLSQP+lq-CMA-ES_Hansen          6  4   0  12   4  11     37
BIPOP-aCMA-STEP_loshchilov      2  5   7   6   4  11     35
SLSQP-scipy-2019_Varelas        1  0   5   5   9  14     34
HMLSL_pal                      16  9   7   1   0   0     33
HE-ES_Glasmachers               0  5  15   6   5   0     31
LSfminbnd_posik                 5  5   5   5   5   5     30
BIPOP-saACM-k_loshchilov        0  0   0   9  11   9     29
SHADE-LM-POP4-to-10_Okulewicz   6  5   5   5   5   0     26
OQNLP_pal                       0  7  12   2   3   0     24
fminunc_pal                     4  9   5   0   5   0     23


### Very synthetic table showing the best algo per dimension, gives info on the year where the algo belongs, its ERT, and it the lowest target such that an ERT is well defined.

In [22]:
import numpy as np
results = []

for dim in sorted(df_all["dimension"].unique()):
    #  Get all rows for this dimension
    df_dim = df_all[df_all["dimension"] == dim]
    
    #  Sort targets from lowest (most precise) to highest
    targets_sorted = sorted(df_dim["target"].unique())
    
    #  Find the lowest target that has at least one finite ERT
    chosen_target = None
    for t in targets_sorted:
        if np.isfinite(df_dim.loc[df_dim["target"] == t, "best_ERT"]).any():
            chosen_target = t
            break
    
    if chosen_target is None:
        # No valid ERTs for this dimension — skip
        continue

    #  Filter to that chosen target and pick the row with the smallest ERT
    df_t = df_dim[df_dim["target"] == chosen_target]
    best_row = df_t.loc[df_t["best_ERT"].idxmin(), ["dimension", "year", "best_algorithm", "target", "best_ERT"]]
    
    results.append(best_row)

#  Build summary DataFrame
df_best_by_dim = pd.DataFrame(results).reset_index(drop=True)

# Rename for clarity
df_best_by_dim.rename(columns={"target": "target_used"}, inplace=True)

print("Best algorithm per dimension (lowest usable target):")
df_best_by_dim.to_csv("results/bbob_lowest_target.csv", index=False)

print(df_best_by_dim)

Best algorithm per dimension (lowest usable target):
   dimension  year                 best_algorithm   target_used  best_ERT
0          2  2021  SHADE-LM-POP4-to-10_Okulewicz  1.000000e-08       4.0
1          3  2021  SHADE-LM-POP4-to-10_Okulewicz  1.000000e-08       5.0
2          5  2021  SHADE-LM-POP4-to-10_Okulewicz  1.000000e-08       7.0
3         10  2021  SHADE-LM-POP4-to-10_Okulewicz  1.000000e-08      12.0
4         20  2021  SHADE-LM-POP4-to-10_Okulewicz  1.000000e-08      22.0
5         40  2019       SLSQP-scipy-2019_Varelas  1.000000e-08      45.0


# 3) Results : 2 different types of tables. 
The goal is to nicely present our results in tables summing up which are the best algorithms over all. 
The definition of “best algorithm” is NOT different between the tables. What is different is the DATA they aggregate.


- We are going to present our results in 2 different ways. A first one being getting the best algo over all dimensions, function AND aggregating over every target precision. 

- A second way is to determine the best algo over all dimensions, functions BUT only for a specific target, and aggregate only on that target. 

## 3.1) Aggregating over every target 



### 3.1.1) Table of the best algos
Here we are building a table that sums up the best algorithms for each dimension: there are 6 different dimensions. We can also see the count for how many times that algorithm was the best in that specific dimension 

The columns of this table are the different dimmension, and the lines of the table represent the ranking of the best algorithms. In the first line we will have the best algorithm for each dimension. The second line will represnt the 2nd best algorithms for each dimension etc... 

In the sence that we are counting wins across all targets at once. Then rank algorithms per dimension across this big mixture.
i.e. “Across ALL target precisions, which algorithm wins the most often in each dimension?”

In [23]:
# === 4. Aggregate: count how many times each algorithm was best per dimension ===
algo_counts = (
    df_best_overall
    .groupby(["dimension", "best_algorithm"])
    .size()
    .reset_index(name="count")
)

# Sort within each dimension by how often each algo was best
algo_counts = algo_counts.sort_values(["dimension", "count"], ascending=[True, False])

# Add ranking per dimension
algo_counts["rank"] = (
    algo_counts
    .groupby("dimension")["count"]
    .rank(method="first", ascending=False)
    .astype(int)
)

# === NEW PART: build a tuple column (algo, count) ===
algo_counts["algo_tuple"] = list(zip(algo_counts["best_algorithm"], algo_counts["count"]))

# === 5. Pivot: rows = rank, columns = dimension, values = (algo_name, count) tuple ===
algo_ranking_table = algo_counts.pivot(index="rank", columns="dimension", values="algo_tuple")

# Sort dimensions in ascending order
algo_ranking_table = algo_ranking_table.reindex(sorted(algo_ranking_table.columns), axis=1)

# === 6. Display neatly ===
print("\n Ranking of best algorithms per dimension (with counts):\n")
from IPython.display import display
display(algo_ranking_table.head(10))  # show top 10 ranks



 Ranking of best algorithms per dimension (with counts):



dimension,2,3,5,10,20,40
rank,,,,,,
1,"(HMLSL_pal, 16)","(lq-CMA-ES_Hansen, 12)","(HE-ES_Glasmachers, 15)","(SLSQP+lq-CMA-ES_Hansen, 12)","(CMAES-APOP-KMA_Nguyen, 12)","(SLSQP-scipy-2019_Varelas, 14)"
2,"(DE-BFGS_voglis, 9)","(HMLSL_pal, 9)","(OQNLP_pal, 12)","(BIPOP-saACM-k_loshchilov, 9)","(BIPOP-saACM-k_loshchilov, 11)","(BIPOP-aCMA-STEP_loshchilov, 11)"
3,"(NELDERDOERR_doerr, 8)","(fminunc_pal, 9)","(BIPOP-aCMA-STEP_loshchilov, 7)","(BIRMIN_Kudela, 8)","(NIPOPaCMA_loshchilov, 10)","(SLSQP+lq-CMA-ES_Hansen, 11)"
4,"(DTS-CMA-ES_Pitra, 7)","(OQNLP_pal, 7)","(HMLSL_pal, 7)","(BIPOP-aCMA-STEP_loshchilov, 6)","(L-BFGS-B-scipy-2019_Varelas, 9)","(NEWUOA_ros, 10)"
5,"(SHADE-LM-POP4-to-10_Okulewicz, 6)","(NELDERDOERR_doerr, 6)","(PSA-CMA-ES_Nishida, 7)","(FULLNEWUOA_ros, 6)","(SLSQP-scipy-2019_Varelas, 9)","(BIPOP-saACM-k_loshchilov, 9)"
6,"(SHADE-LM_Okulewicz, 6)","(BIPOP-aCMA-STEP_loshchilov, 5)","(SLSQP-11-scipy_Hansen, 6)","(HE-ES_Glasmachers, 6)","(CMA-ES-Akimoto_Gharafi, 5)","(CMAES-APOP-KMA_Nguyen, 7)"
7,"(SLSQP+lq-CMA-ES_Hansen, 6)","(HE-ES_Glasmachers, 5)","(lq-CMA-ES_Hansen, 6)","(CMAES-APOP-MA_Nguyen, 5)","(HE-ES_Glasmachers, 5)","(NIPOPaCMA_loshchilov, 6)"
8,"(LSfminbnd_posik, 5)","(I-DBDP-GL_Kudela, 5)","(DEctpb_posik, 5)","(DEctpb_posik, 5)","(LSfminbnd_posik, 5)","(SLSQP-11-scipy_Hansen, 6)"
9,"(Powell-scipy-2019_Varelas, 5)","(LSfminbnd_posik, 5)","(LSfminbnd_posik, 5)","(HCMA_loshchilov, 5)","(SHADE-LM-POP4-to-10_Okulewicz, 5)","(LSfminbnd_posik, 5)"


We have noticed that the best algorithms come as batches of 6 algorithms per line. So hence when we are asking for the N best algorithms, where N<6, it is complicated to choose which algorithms to return. This is why we will set a minimum of 6 best algorithms, one for each dimension. Now we may be asked for more than 6 best algrithms. 


For example let's say we are asking for the 8 best algorithms. In this case we will go through the following lines of the table. There are several cases we need to consider to choose these 8 best algorithms.

- One scenario, is that in the following line, there are the names of algorithms that were already mentioned in the first line. In this case we don't add their name again. 
- The second scenario is we are in fact not able to return the names of 8 algorithms, because all the algorithms have performed equally for their respective dimension. Instead of choosing only 8, we will return the names of all the algorithms that are mentioned and are on the same line. We also notify that we are not returning 8 algorithms, but a bit more, here 10.
- A further step would be to really restrict the number to 8. This means we need to find a way to classify the algorithms that are on the same line. An easy way is to select the one that has the highest occurence of "best" algorithm. 

### 3.1.2) List of the best algos according to the table

Here our goal is to return N number of best algo depending on how many best algo the user is asking for. We will treat every algo on a same row equally, not one being better than the other. 

In [24]:
def get_top_algorithms(algo_ranking_table, N_min):
    """
    Collect algorithms from top ranks (rows) across all dimensions
    until at least N_min unique algorithms are found.
    """
    seen_algos = set()
    row_idx = 0

    # Keep adding rows until we have at least N_min unique algorithms
    while len(seen_algos) < N_min and row_idx < len(algo_ranking_table):
        row_algos = algo_ranking_table.iloc[row_idx].dropna().unique()

        # Extract only the algorithm names (first element of tuple)
        algo_names = {algo_tuple[0] for algo_tuple in row_algos}

        # Add these clean names to the seen set
        seen_algos.update(algo_names)
        row_idx += 1

    algo_list = sorted(seen_algos)

    print(f"\n Requested {N_min} best algorithms.")
    print(f" Returning {len(algo_list)} unique algorithms (reached rank {row_idx}).\n")
    print(" List of selected algorithms:\n")
    for algo in algo_list:
        print(f" - {algo}")

    return algo_list


# === Interactive part ===
try:
    # Ask user for number of desired algorithms
    N_input = int(input("How many best algorithms do you want? "))
    if N_input < 1:
        print(" Minimum number of best algorithms is 1. Using N=1.")
        N_input = 1

    # Compute and display
    best_algos = get_top_algorithms(algo_ranking_table, N_min=N_input)

except ValueError:
    print("Invalid input. Please enter an integer number (e.g., 6 or 7).")


How many best algorithms do you want? 1

 Requested 1 best algorithms.
 Returning 6 unique algorithms (reached rank 1).

 List of selected algorithms:

 - CMAES-APOP-KMA_Nguyen
 - HE-ES_Glasmachers
 - HMLSL_pal
 - SLSQP+lq-CMA-ES_Hansen
 - SLSQP-scipy-2019_Varelas
 - lq-CMA-ES_Hansen


## 3.2) Choice of a specific target precision for the best algo.

### 3.2.1) Table dpending on a specific target precision


- We filter by ONE target precision of our choice.

- Count wins only for that target.

- Rank algos per dimension based only on those wins. 

i.e. “For target = X, which algorithm wins the most functions in each dimension?”

## 3.2)  Best algo depending on a specific target precision


- We filter by ONE target precision of our choice.

- Count wins only for that target.

- Rank algos per dimension based only on those wins. 

i.e. “For target = X, which algorithm wins the most functions in each dimension?”

In [25]:
import pandas as pd
from pathlib import Path

# --- assuming df_best_overall is already built as before ---
# columns: ["dimension", "function_id", "year", "best_algorithm", "target", "best_ERT"]

def make_dimension_ranking_table(df_best_overall, target):
    """
    For a given target:
    - Count, for each dimension, how many times each algorithm was best.
    - Rank algorithms within each dimension by this count.
    - Pivot to get a table: rows = rank, columns = dimension, values = algorithm name.
    """
    df_t = df_best_overall[df_best_overall["target"] == target].copy()
    if df_t.empty:
        raise ValueError(f"No data found for target = {target}")

    # Aggregate: how many times each algorithm is best per dimension
    algo_counts = (
        df_t.groupby(["dimension", "best_algorithm"])
        .size()
        .reset_index(name="count")
    )

    # Sort within each dimension by count
    algo_counts = algo_counts.sort_values(
        ["dimension", "count"], ascending=[True, False]
    )

    # Add rank per dimension
    algo_counts["rank"] = (
        algo_counts.groupby("dimension")["count"]
        .rank(method="first", ascending=False)
        .astype(int)
    )

    # Pivot: rows = rank, columns = dimension
    ranking_table = algo_counts.pivot(
        index="rank", columns="dimension", values="best_algorithm"
    )

    ranking_table = ranking_table.reindex(
        sorted(ranking_table.columns),
        axis=1
    )

    return ranking_table


In [26]:
def get_top_algorithms_from_rank_table(rank_table, N_min):
    """
    Take the dimension×rank table:
    - Start from rank 1, then rank 2, etc.
    - Collect all algorithms that appear in each row (all dimensions).
    - Stop when we have at least N_min unique algorithms.
    """
    seen = set()
    row_idx = 0

    while len(seen) < N_min and row_idx < len(rank_table):
        row_algos = rank_table.iloc[row_idx].dropna().unique()
        seen.update(row_algos)
        row_idx += 1

    algo_list = sorted(seen)

    print(f"\nRequested: {N_min} algorithms")
    print(f"Returning: {len(algo_list)} unique algorithms (used ranks 1–{row_idx}).\n")
    print("Selected algorithms:")
    for a in algo_list:
        print(" -", a)

    return algo_list


In [27]:
def get_top_algorithms_from_rank_table(rank_table, N_min):
    """
    From the dimension×rank table:
    - Iterate rank rows in order (1,2,3,...).
    - Collect algorithms left → right (dimension order).
    - Preserve order of first appearance.
    - Stop when at least N_min unique algorithms are collected.
    """

    ordered_algos = []     # keeps order of appearance
    seen = set()           # for fast membership check

    row_idx = 0

    while len(seen) < N_min and row_idx < len(rank_table):
        row = rank_table.iloc[row_idx]

        # Loop across dimensions in display order
        for algo in row.dropna():
            if algo not in seen:
                seen.add(algo)
                ordered_algos.append(algo)

        row_idx += 1

    print(f"\nRequested: {N_min} algorithms")
    print(f"Returning: {len(ordered_algos)} algorithms (used ranks 1–{row_idx}).\n")

    print("Selected algorithms (ordered by ranking row):")
    for algo in ordered_algos:
        print(" -", algo)

    return ordered_algos


In [29]:
try:
    print("Available targets:", df_best_overall["target"].unique())

    target_input = float(input("\nSelect a target precision (e.g., 1e-8 or 1e-5): "))

    algo_ranking_table = make_dimension_ranking_table(df_best_overall, target_input)
    print(f"\nRanking table built for target = {target_input}")
    from IPython.display import display
    display(algo_ranking_table.head(10))

    N_input = int(input("\nHow many best algorithms do you want?: "))
    if N_input < 1:
        print("Minimum number of best algorithms is 1. Using N = 1.")
        N_input = 1

    best_algos = get_top_algorithms_from_rank_table(algo_ranking_table, N_input)

except ValueError:
    print("Invalid input: please enter numeric values for target and N.")


Available targets: [1.e-08 1.e-05 1.e-03 1.e-02 1.e-01]

Select a target precision (e.g., 1e-8 or 1e-5): 1e-8

Ranking table built for target = 1e-08


dimension,2,3,5,10,20,40
rank,,,,,,
1,DE-BFGS_voglis,lq-CMA-ES_Hansen,HE-ES_Glasmachers,BIPOP-saACM-k_loshchilov,BIPOP-saACM-k_loshchilov,BIPOP-saACM-k_loshchilov
2,DTS-CMA-ES_Pitra,GLOBAL_pal,BIPOP-aCMA-STEP_loshchilov,HCMA_loshchilov,CMAES-APOP-KMA_Nguyen,BIPOP-aCMA-STEP_loshchilov
3,HMLSL_pal,NELDER_hansen,GLOBAL_pal,BIPOPsaACM_loshchilov,NIPOPaCMA_loshchilov,CMAES-APOP-KMA_Nguyen
4,NELDERDOERR_doerr,BIPOP-aCMA-STEP_loshchilov,PSA-CMA-ES_Nishida,FULLNEWUOA_ros,BIPOP-aCMA-STEP_loshchilov,NEWUOA_ros
5,SHADE-LM-POP4-to-10_Okulewicz,DASA_korosec,lq-CMA-ES_Hansen,PSA-CMA-ES_Nishida,CMA-ES-Akimoto_Gharafi,NIPOPaCMA_loshchilov
6,SHADE-LM_Okulewicz,DE-BFGS_voglis,CMAES-APOP-Var1_Nguyen,SLSQP+lq-CMA-ES_Hansen,COBYLA-scipy-2019_Varelas,BFGS-P-09_Blelly
7,fmincon_pal,DE-scipy-2019_Varelas,DEctpb_posik,BIPOP-aCMA-STEP_loshchilov,HCMA_loshchilov,CMA-ES-Akimoto_Gharafi
8,BFGS-P-StPt_Blelly,DTS-CMA-ES_Pitra,DTS-CMA-ES_Pitra,BIRMIN_Kudela,HE-ES_Glasmachers,CMAES_Hutter_hutter
9,DIRECT-REV_Kudela,HE-ES_Glasmachers,HMLSL_pal,CMAES-APOP-MA_Nguyen,IPOPsaACM_loshchilov,LSfminbnd_posik



How many best algorithms do you want?: 3

Requested: 3 algorithms
Returning: 4 algorithms (used ranks 1–1).

Selected algorithms (ordered by ranking row):
 - DE-BFGS_voglis
 - lq-CMA-ES_Hansen
 - HE-ES_Glasmachers
 - BIPOP-saACM-k_loshchilov


### Other view with both the count and the rank 

In [15]:
import pandas as pd

# --- assuming df_best_overall is already built as before ---
# columns: ["dimension", "function_id", "year", "best_algorithm", "target", "best_ERT"]


def make_dimension_ranking_table_with_counts(df_best_overall, target):
    """
    For a given target:
    - Count, for each dimension, how many times each algorithm was best.
    - Rank algorithms within each dimension by this count.
    - Pivot into a table: rows = rank, columns = dimension,
      values = (algorithm, count) tuples.
    """
    # Filter by target
    df_t = df_best_overall[df_best_overall["target"] == target].copy()
    if df_t.empty:
        raise ValueError(f"No data found for target = {target}")

    # Aggregate: count how many times each algorithm is best in each dimension
    algo_counts = (
        df_t.groupby(["dimension", "best_algorithm"])
        .size()
        .reset_index(name="count")
    )

    # Sort inside each dimension
    algo_counts = algo_counts.sort_values(
        ["dimension", "count"], ascending=[True, False]
    )

    # Rank per dimension
    algo_counts["rank"] = (
        algo_counts.groupby("dimension")["count"]
        .rank(method="first", ascending=False)
        .astype(int)
    )

    # Build tuple column
    algo_counts["algo_tuple"] = list(
        zip(algo_counts["best_algorithm"], algo_counts["count"])
    )

    # Pivot: rows = rank, columns = dimension, values = tuple
    ranking_table = algo_counts.pivot(
        index="rank", columns="dimension", values="algo_tuple"
    )

    # Sort dimensions
    ranking_table = ranking_table.reindex(
        sorted(ranking_table.columns), axis=1
    )

    return ranking_table


# === Interactive part: only ask for target ===
try:
    print("Available targets:", df_best_overall["target"].unique())

    # User input
    target_input = float(input("\nSelect a target precision (e.g., 1e-8 or 1e-5): "))

    # Build the table
    algo_ranking_table = make_dimension_ranking_table_with_counts(
        df_best_overall,
        target_input
    )

    print(f"\nRanking table built for target = {target_input}")
    from IPython.display import display
    display(algo_ranking_table.head(3))

except ValueError:
    print("Invalid target input. Please enter a numeric target.")


Available targets: [1.e-08 1.e-05 1.e-03 1.e-02 1.e-01]

Select a target precision (e.g., 1e-8 or 1e-5): 1e-8 

Ranking table built for target = 1e-08


dimension,2,3,5,10,20,40
rank,,,,,,
1,"(DE-BFGS_voglis, 2)","(lq-CMA-ES_Hansen, 3)","(HE-ES_Glasmachers, 3)","(BIPOP-saACM-k_loshchilov, 3)","(BIPOP-saACM-k_loshchilov, 6)","(BIPOP-saACM-k_loshchilov, 5)"
2,"(DTS-CMA-ES_Pitra, 2)","(GLOBAL_pal, 2)","(BIPOP-aCMA-STEP_loshchilov, 2)","(HCMA_loshchilov, 3)","(CMAES-APOP-KMA_Nguyen, 3)","(BIPOP-aCMA-STEP_loshchilov, 3)"
3,"(HMLSL_pal, 2)","(NELDER_hansen, 2)","(GLOBAL_pal, 2)","(BIPOPsaACM_loshchilov, 2)","(NIPOPaCMA_loshchilov, 3)","(CMAES-APOP-KMA_Nguyen, 2)"


# 4) Ploting the results of table 1 ( aggregate over all dimension) 

In [13]:
import cocopp

# run postprocessing
cocopp.main(['CMAES-APOP-KMA_Nguyen','HE-ES_Glasmachers','HMLSL_pal','bbob/2020/SLSQP+lq-CMA-ES_Hansen.tgz','SLSQP-scipy-2019_Varelas','bbob/2020/lq-CMA-ES_Hansen.tgz'])


Post-processing (2+)
  Using 6 data sets:
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2022\CMAES-APOP-KMA_Nguyen.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2020\HE-ES_Glasmachers.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2013\HMLSL_pal_noiseless.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2020\SLSQP+lq-CMA-ES_Hansen.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2019\SLSQP-scipy-2019_Varelas.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2020\lq-CMA-ES_Hansen.tgz

Post-processing (2+)
  loading data...
  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2022\CMAES-APOP-KMA_Nguyen.tgz
  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2020\HE-ES_Glasmachers.tgz
  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2013\HMLSL_pal_noiseless.tgz
  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\

DictAlg([(('CMAES-APOP-KMA_Nguyen', ''),
          [DataSet(CMAES-APOP-KMA_Nguyen on f1 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f2 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f3 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f4 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f5 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f6 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f7 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f8 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f9 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f10 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f11 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f12 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f13 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f14 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f15 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f16 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f17 2-D),
           DataSet(CMAES-APOP-KMA_Nguyen on f18 2-

# 5) Plotting the result of the best algo for a given target.

In [17]:
cocopp.main(['DE-BFGS_voglis', 'bbob/2020/lq-CMA-ES_Hansen.tgz','HE-ES_Glasmachers', 'BIPOP-saACM-k_loshchilov'])

Post-processing (2+)
  Using 4 data sets:
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2012\DE-BFGS_voglis_noiseless.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2020\lq-CMA-ES_Hansen.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2020\HE-ES_Glasmachers.tgz
    C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2013\BIPOP-saACM-k_loshchilov_noiseless.tgz

Post-processing (2+)
  loading data...
  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2012\DE-BFGS_voglis_noiseless.tgz
  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2020\lq-CMA-ES_Hansen.tgz
  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2020\HE-ES_Glasmachers.tgz
  using: C:\Users\elsaf\AppData\Local\cocopp\cocopp\Cache\das\da\bbob\2013\BIPOP-saACM-k_loshchilov_noiseless.tgz
  Will generate output data in folder ppdata\DE-BF_lq-CM_HE-ES_BIPOP_112700h3121
    this might take several minutes

DictAlg([(('DE-BFGS_voglis', ''),
          [DataSet(DE-BFGS_voglis on f1 2-D),
           DataSet(DE-BFGS_voglis on f2 2-D),
           DataSet(DE-BFGS_voglis on f3 2-D),
           DataSet(DE-BFGS_voglis on f4 2-D),
           DataSet(DE-BFGS_voglis on f5 2-D),
           DataSet(DE-BFGS_voglis on f6 2-D),
           DataSet(DE-BFGS_voglis on f7 2-D),
           DataSet(DE-BFGS_voglis on f8 2-D),
           DataSet(DE-BFGS_voglis on f9 2-D),
           DataSet(DE-BFGS_voglis on f10 2-D),
           DataSet(DE-BFGS_voglis on f11 2-D),
           DataSet(DE-BFGS_voglis on f12 2-D),
           DataSet(DE-BFGS_voglis on f13 2-D),
           DataSet(DE-BFGS_voglis on f14 2-D),
           DataSet(DE-BFGS_voglis on f15 2-D),
           DataSet(DE-BFGS_voglis on f16 2-D),
           DataSet(DE-BFGS_voglis on f17 2-D),
           DataSet(DE-BFGS_voglis on f18 2-D),
           DataSet(DE-BFGS_voglis on f19 2-D),
           DataSet(DE-BFGS_voglis on f20 2-D),
           DataSet(DE-BFGS_voglis o

In [ ]:
1	(DE-BFGS_voglis, 2)	(lq-CMA-ES_Hansen, 3)	(HE-ES_Glasmachers, 3)	(BIPOP-saACM-k_loshchilov, 3)	(BIPOP-saACM-k_loshchilov, 6)	(BIPOP-saACM-k_loshchilov, 5)